# 第一組：坡度變化渠道的三種計算方法
本筆記本計算長30 m、寬0.46 m的矩形渠道：上游陡、下游緩，固定總流量，入口採正常水深，出口採指定尾水。先閱讀 [問題設定](docs/problem.html) 與 [方程式講義](docs/methods.html)。
SS是Standard-step method（標準步法）；SV是Saint-Venant（聖維南淺水方程）；B是本套件的定常壓力近似Boussinesq型模型。B的波動仍待研究。
水面圖統一使用 $\eta=z_b+h$；流速與流量關係為 $q=Q/W=hu$。P0是共同基準，每人再做自己的低、高水準。
## 步驟1：選案例並讀參數
下方選P0開始。`S1/S2`是兩段床坡，`n`是Manning粗糙係數，`hin_m/hout_m`是兩端水深，`Fr_in`是入口福祿數。Shift+Enter執行後再往下。

In [ ]:
from pathlib import Path
import json
import lab
from postprocess import plot_compare
from IPython.display import display, Image
STUDENT_ID = "A1"
CASE_ID = "P0"
case = lab.load_case(CASE_ID)
{k: case[k] for k in ["id", "Q_m3s", "S1", "S2", "n", "hin_m", "hout_m", "Fr_in"]}

## 步驟2：執行共同基準
P0採原uniform初場以便重現；個人變體採tanh平滑初場。SS斷面間距0.025 m，SV用512單元，B用1024區間；這是第一次操作，不是全部網格比較。
每次都會新建`runs`資料夾。請等待完成並保留資料。

In [ ]:
INITIAL = "uniform" if CASE_ID == "P0" else "tanh"
lab.build()
ss_run = lab.run_method(case, "ss", dx=0.025)
sv_run = lab.run_method(case, "sv", N=512, initial=INITIAL)
b_run = lab.run_method(case, "b", N=1024, initial=INITIAL)

## 步驟3：比較水面高程
下圖畫 $\eta=z_b+h$，附渠底；完整圖查邊界，局部圖看水躍與波動。SS是定常跳接、SV是200–300秒平均，B是固定點續算後水面。不同狀態不要混稱同一瞬間。

In [ ]:
figure = plot_compare([ss_run, sv_run, b_run], b_run / "comparison")
display(Image(filename=str(figure)))

## 步驟4：讀指標而不只看圖
`x50_unique_m`是唯一的50%參考水深上穿位置；多個交點時為null，須查看`all_upcrossings_m`。`first_peak_h_m`是水深峰值，`first_peak_x_m`是位置；尾波峰距與衰減見 [結果判讀](docs/results.html)。null不是零。

In [ ]:
summary = json.loads((b_run / "summary.json").read_text())
{k: summary[k] for k in ["x50_unique_m", "all_upcrossings_m", "first_peak_h_m", "first_peak_x_m", "tail_wavelength_m", "tail_amplitude_efold_m"]}

## 步驟5：個人案例與四項控制
先把CASE_ID改成你的PnL/PnH，按 [第一組操作](docs/group_a.html) 完成1024、2048主要計算，再選一個變體做下列控制。CFL控制時間步，$\lambda$控制數值耗散；比較1024、$\lambda=1$與2048、$\lambda=2$近似固定$\lambda\Delta x$。B off關閉同核心非靜水壓修正。
下方開關預設False；確認案例後才改True。

In [ ]:
RUN_SENSITIVITY = False
if RUN_SENSITIVITY:
    sv_half = lab.run_method(case, "sv", N=2048, CFL=0.2)
    b_half = lab.run_method(case, "b", N=2048, CFL=0.05)
    b_dissipation = lab.run_method(case, "b", N=2048, lam=2)
    b_off = lab.run_method(case, "b", N=2048, B=0)

## 下一步與繳交
依 [繳交格式](docs/submission.html) 保存參數、原始結果、圖與指標。完成數值重現不代表波動已收斂；沒有唯一指標時，說明N/A原因。